# Interaction evidence and Evidence Map debug

This notebook is a focused audit notebook for multi-drug questions. It follows the backend pieces that happen after query understanding: secondary evidence selection, regular and interaction-targeted OpenFDA evidence, evidence merging, and the question-level Evidence Map.

Use it to inspect whether a label source in the Evidence Map actually exists in Supporting Evidence, which medication concepts it is connected to, which provenance tags it carries, and which interaction lookup terms produced it.

## 0. Setup

Run from either the repository root or the `notebooks/` directory. The OpenFDA client logs every live request URL in the terminal/notebook output, which is useful for checking interaction-search behavior.

In [ ]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repository root.")


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
logging.getLogger("src.dossier.openfda_store").setLevel(logging.INFO)
logging.getLogger("src.query_answer.service").setLevel(logging.INFO)
logging.getLogger("src.query_understanding.service").setLevel(logging.INFO)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

print(f"Repository root: {ROOT}")

In [ ]:
from src.dossier.builder import DossierBuilder
from src.dossier.models import OpenFDALabelEvidence, OpenFDALabelRecord
from src.dossier.openfda_store import OpenFDALabelStore
from src.dossier.rxnorm_store import RxNormParquetStore
from src.query_answer.config import load_query_answer_parameters
from src.query_answer.evidence_map import (
    INTERACTION_TARGETED_TAG,
    build_question_evidence_map,
)
from src.query_answer.secondary import (
    build_secondary_evidence,
    select_secondary_mentions,
    stable_record_key,
)
from src.query_understanding.extractor import HybridQueryExtractor
from src.query_understanding.service import QueryUnderstandingService


def to_jsonable(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    return value


def show_json(value):
    display(JSON(to_jsonable(value)))


def as_frame(rows):
    return pd.DataFrame([to_jsonable(row) for row in rows])

## 1. Query and runtime settings

This query is the current debugging example. Running query understanding may call the query-extraction LLM when the extraction environment variables are configured. Answer synthesis is intentionally not called in this notebook.

In [ ]:
QUERY = "I currently take cetirizine for my pollen allergy. can i take both ibuprofen and aspirin against swollen eyes?"

parameters = load_query_answer_parameters()
OPENFDA_LIMIT = parameters.default_openfda_limit

print(f"Query: {QUERY}")
print(parameters)
print("Query extraction LLM configured:", bool(os.getenv("QUERY_EXTRACTION_OPENAI_API_KEY") and os.getenv("QUERY_EXTRACTION_OPENAI_MODEL")))
print("Answer synthesis is not called in this notebook.")

## 2. Instantiate the same backend services

This mirrors the FastAPI path: `DossierBuilder` owns the RxNorm parquet store and OpenFDA label store; `QueryUnderstandingService` extracts state and resolves drug mentions.

In [ ]:
rxnorm_store = RxNormParquetStore()
openfda_store = OpenFDALabelStore(use_cache=True, allow_live=True)
builder = DossierBuilder(rxnorm_store=rxnorm_store, openfda_store=openfda_store)

extractor = HybridQueryExtractor()
understanding_service = QueryUnderstandingService(builder=builder, extractor=extractor)

print("Services ready.")

## 3. Query understanding: extracted state and resolved mentions

This is the first place to check whether unexpected medications entered the pipeline. Only medications present in the extracted state should be eligible for secondary evidence tabs.

In [ ]:
understanding = understanding_service.understand(QUERY, openfda_limit=OPENFDA_LIMIT)

display(Markdown("### Extracted state"))
show_json(understanding.state)

display(Markdown("### Resolved drug mentions"))
resolved_rows = []
for mention in understanding.resolved_drugs:
    concept = mention.selected_concept
    resolved_rows.append(
        {
            "text": mention.text,
            "role": mention.role,
            "selected_name": concept.name if concept else None,
            "selected_rxcui": concept.rxcui if concept else None,
            "selected_tty": concept.tty if concept else None,
            "candidate_count": len(mention.candidates),
            "top_candidate_score": mention.candidates[0].score if mention.candidates else None,
        }
    )
pd.DataFrame(resolved_rows)

## 4. Primary dossier label evidence

The primary dossier is still the only full dossier in V1. This table shows the actual OpenFDA label records that should be visible in the primary Supporting Evidence tab.

In [ ]:
primary = understanding.primary_dossier.resolved_drug if understanding.primary_dossier else None
primary_evidence = understanding.primary_dossier.label_evidence if understanding.primary_dossier else None

print("Primary concept:", primary)
print("Primary label evidence:", None if primary_evidence is None else {
    "rxcui": primary_evidence.rxcui,
    "retrieval_mode": primary_evidence.retrieval_mode,
    "labels_found": primary_evidence.labels_found,
    "label_limit": primary_evidence.label_limit,
    "errors": primary_evidence.errors,
})

In [ ]:
def record_rows(evidence: OpenFDALabelEvidence | None, *, bundle: str) -> list[dict]:
    if evidence is None:
        return []
    rows = []
    for index, record in enumerate(evidence.label_records, start=1):
        rows.append(
            {
                "bundle": bundle,
                "record_index": index,
                "source_id": record.source_id,
                "stable_key": stable_record_key(record),
                "brand": ", ".join(record.brand_names),
                "generic": ", ".join(record.generic_names),
                "manufacturer": ", ".join(record.manufacturer_names),
                "rxcuis_on_label": ", ".join(record.rxcuis),
                "tags": ", ".join(record.provenance_tags),
                "route": ", ".join(record.routes),
                "product_type": ", ".join(record.product_types),
            }
        )
    return rows


def section_rows(evidence: OpenFDALabelEvidence | None, *, bundle: str) -> list[dict]:
    if evidence is None:
        return []
    rows = []
    for section_name, entries in evidence.sections.items():
        for entry_index, entry in enumerate(entries, start=1):
            rows.append(
                {
                    "bundle": bundle,
                    "section": section_name,
                    "entry_index": entry_index,
                    "source_id": entry.source_id,
                    "tags": ", ".join(entry.provenance_tags),
                    "text_preview": entry.text[:220],
                }
            )
    return rows


primary_records_df = pd.DataFrame(record_rows(primary_evidence, bundle="primary"))
primary_sections_df = pd.DataFrame(section_rows(primary_evidence, bundle="primary"))
primary_records_df

## 5. Which secondary mentions are eligible?

`select_secondary_mentions` is the gatekeeper for secondary tabs/evidence. It only uses resolved current/mentioned medications whose text or resolved concept is present in the extracted state.

In [ ]:
secondary_mentions = select_secondary_mentions(
    understanding.resolved_drugs,
    understanding.state,
    primary,
    max_items=parameters.max_secondary_drugs,
) if primary else []

pd.DataFrame(
    [
        {
            "mention_text": mention.text,
            "role": mention.role,
            "resolved_name": mention.selected_concept.name if mention.selected_concept else None,
            "resolved_rxcui": mention.selected_concept.rxcui if mention.selected_concept else None,
            "resolved_tty": mention.selected_concept.tty if mention.selected_concept else None,
        }
        for mention in secondary_mentions
    ]
)

## 6. Build secondary evidence

This performs the regular secondary OpenFDA lookup and, when `interaction_check` is one of the extracted intents, the interaction-targeted lookup in both directions. The OpenFDA URLs are logged above/below this cell.

In [ ]:
secondary_evidence = build_secondary_evidence(understanding, builder, parameters)

pd.DataFrame(
    [
        {
            "mention_text": item.mention_text,
            "role": item.role,
            "resolved_name": item.resolved_concept.name,
            "resolved_rxcui": item.resolved_concept.rxcui,
            "resolved_tty": item.resolved_concept.tty,
            "retrieval_modes": ", ".join(item.retrieval_modes),
            "merged_labels_found": item.label_evidence.labels_found if item.label_evidence else None,
            "interaction_labels_found": item.interaction_label_evidence.labels_found if item.interaction_label_evidence else None,
            "merged_errors": " | ".join(item.label_evidence.errors) if item.label_evidence else None,
            "interaction_errors": " | ".join(item.interaction_label_evidence.errors) if item.interaction_label_evidence else None,
            "rxnorm_context": item.rxnorm_context.status if item.rxnorm_context else None,
        }
        for item in secondary_evidence
    ]
)

## 7. Secondary records: merged evidence vs interaction-only evidence

The merged evidence is what the frontend Supporting Evidence tab receives. The interaction-only evidence is useful for checking whether interaction-specific records were later deduplicated into the merged bundle.

In [ ]:
secondary_record_rows = []
secondary_section_rows = []
interaction_only_record_rows = []
interaction_only_section_rows = []

for item in secondary_evidence:
    label = f"secondary:{item.resolved_concept.name}"
    interaction_label = f"interaction-only:{item.resolved_concept.name}"
    secondary_record_rows.extend(record_rows(item.label_evidence, bundle=label))
    secondary_section_rows.extend(section_rows(item.label_evidence, bundle=label))
    interaction_only_record_rows.extend(record_rows(item.interaction_label_evidence, bundle=interaction_label))
    interaction_only_section_rows.extend(section_rows(item.interaction_label_evidence, bundle=interaction_label))

secondary_records_df = pd.DataFrame(secondary_record_rows)
secondary_sections_df = pd.DataFrame(secondary_section_rows)
interaction_only_records_df = pd.DataFrame(interaction_only_record_rows)
interaction_only_sections_df = pd.DataFrame(interaction_only_section_rows)

display(Markdown("### Merged secondary label records"))
display(secondary_records_df)
display(Markdown("### Interaction-only label records before final merge"))
display(interaction_only_records_df)

## 8. Interaction-specific rows and possible owner ambiguity

These rows are the ones that should become dashed interaction-specific nodes/edges in the Evidence Map. Check the `rxcuis_on_label` and names carefully: a label can be retrieved while asking about one medication but actually belong to another medication's FDA label.

In [ ]:
all_records_df = pd.concat(
    [primary_records_df, secondary_records_df],
    ignore_index=True,
) if not primary_records_df.empty or not secondary_records_df.empty else pd.DataFrame()

if all_records_df.empty:
    display(Markdown("No records found."))
else:
    interaction_records_df = all_records_df[
        all_records_df["tags"].str.contains(INTERACTION_TARGETED_TAG, na=False)
    ].copy()
    display(interaction_records_df)

    display(Markdown("### Duplicate source IDs across bundles"))
    duplicated_sources = all_records_df[
        all_records_df.duplicated("source_id", keep=False)
    ].sort_values(["source_id", "bundle"])
    display(duplicated_sources)

## 9. Build the Question Evidence Map

This is the exact object the frontend renders. The next cells flatten the nodes and edges for inspection.

In [ ]:
evidence_map = build_question_evidence_map(
    understanding,
    secondary_evidence=secondary_evidence,
)

nodes_df = pd.DataFrame([node.model_dump(mode="json") for node in evidence_map.nodes])
edges_df = pd.DataFrame([edge.model_dump(mode="json") for edge in evidence_map.edges])

display(Markdown("### Evidence map summary counts"))
show_json(evidence_map.summary_counts)
display(Markdown("### Nodes"))
display(nodes_df.sort_values(["kind", "label"]))
display(Markdown("### Edges"))
display(edges_df.sort_values(["kind", "source", "target"]))

## 10. Evidence Map interaction edges

Every `interaction_lookup_source` edge should identify the two medication terms used for that interaction-specific lookup. This is the clearest place to catch missing or wrong interaction context.

In [ ]:
node_lookup = nodes_df.set_index("id").to_dict("index") if not nodes_df.empty else {}


def node_label(node_id: str) -> str:
    node = node_lookup.get(node_id, {})
    return node.get("label") or node_id


interaction_edges = edges_df[edges_df["kind"] == "interaction_lookup_source"].copy() if not edges_df.empty else pd.DataFrame()
if not interaction_edges.empty:
    interaction_edges["source_label"] = interaction_edges["source"].map(node_label)
    interaction_edges["target_label"] = interaction_edges["target"].map(node_label)
display(interaction_edges[[
    "source_label",
    "target_label",
    "label",
    "source_id",
    "rxcui",
    "interaction_terms",
    "tags",
]] if not interaction_edges.empty else interaction_edges)

## 11. Map source nodes vs Supporting Evidence records

A label source node should not exist in the map unless the same `source_id` exists in primary or secondary label evidence. If `missing_from_supporting_evidence` is true, the graph can navigate to a node that the UI cannot show.

In [ ]:
supporting_source_ids = set(all_records_df["source_id"].dropna()) if not all_records_df.empty else set()
source_nodes = nodes_df[nodes_df["kind"] == "label_source"].copy() if not nodes_df.empty else pd.DataFrame()
if not source_nodes.empty:
    source_nodes["missing_from_supporting_evidence"] = ~source_nodes["source_id"].isin(supporting_source_ids)
    source_nodes["matching_record_bundles"] = source_nodes["source_id"].map(
        lambda source_id: sorted(all_records_df.loc[all_records_df["source_id"] == source_id, "bundle"].unique())
        if not all_records_df.empty else []
    )
display(source_nodes[[
    "id",
    "label",
    "subtitle",
    "rxcui",
    "label_rxcuis",
    "source_id",
    "evidence_scope",
    "tags",
    "missing_from_supporting_evidence",
    "matching_record_bundles",
]] if not source_nodes.empty else source_nodes)

## 12. Source connectivity audit

This condenses the graph into one row per label source. It shows which medication nodes connect through normal ownership edges and which medication nodes connect through interaction-specific lookup edges.

In [ ]:
connectivity_rows = []
for _, source_node in source_nodes.iterrows() if not source_nodes.empty else []:
    incoming = edges_df[edges_df["target"] == source_node["id"]]
    owner_edges = incoming[incoming["kind"] == "has_label_source"]
    lookup_edges = incoming[incoming["kind"] == "interaction_lookup_source"]
    section_edges = edges_df[
        (edges_df["source"] == source_node["id"]) & (edges_df["kind"] == "has_label_section")
    ]
    connectivity_rows.append(
        {
            "source_label": source_node["label"],
            "source_id": source_node["source_id"],
            "source_node_rxcui": source_node["rxcui"],
            "openfda_label_rxcuis": source_node["label_rxcuis"],
            "tags": source_node["tags"],
            "normal_owner_medications": sorted(owner_edges["source"].map(node_label).unique()),
            "interaction_lookup_medications": sorted(lookup_edges["source"].map(node_label).unique()),
            "interaction_terms": sorted({
                term
                for terms in lookup_edges["interaction_terms"]
                for term in (terms or [])
            }),
            "section_count": len(section_edges),
            "sections": sorted(section_edges["section"].dropna().unique()),
        }
    )

pd.DataFrame(connectivity_rows).sort_values(["source_label", "source_id"]) if connectivity_rows else pd.DataFrame()

## 13. Label sections represented in the map

This catches section nodes that cannot be shown in the UI because their source or section is missing from the underlying evidence object.

In [ ]:
all_sections_df = pd.concat(
    [primary_sections_df, secondary_sections_df],
    ignore_index=True,
) if not primary_sections_df.empty or not secondary_sections_df.empty else pd.DataFrame()

section_nodes = nodes_df[nodes_df["kind"] == "label_section"].copy() if not nodes_df.empty else pd.DataFrame()
if not section_nodes.empty:
    support_pairs = set(
        zip(
            all_sections_df["source_id"],
            all_sections_df["section"],
        )
    ) if not all_sections_df.empty else set()
    section_nodes["missing_section_from_supporting_evidence"] = [
        (row.source_id, row.section) not in support_pairs
        for row in section_nodes.itertuples()
    ]
display(section_nodes[[
    "label",
    "source_id",
    "section",
    "rxcui",
    "label_rxcuis",
    "tags",
    "missing_section_from_supporting_evidence",
]] if not section_nodes.empty else section_nodes)

## 14. Raw JSON snapshots

Use these when a table hides too much detail. They are intentionally at the end so the earlier cells stay readable.

In [ ]:
display(Markdown("### Secondary evidence JSON"))
show_json(secondary_evidence)
display(Markdown("### Evidence map JSON"))
show_json(evidence_map)